# Bank Marketing — Model Training

## Описание этапа

**Цель:** обучить и настроить модели бинарной классификации для прогнозирования положительного отклика клиента на предложение открыть срочный депозит.

На предыдущем этапе (`02_preprocessing.ipynb`) был подготовлен пайплайн предобработки, объединяющий несколько способов кодирования и масштабирования признаков.

В этом ноутбуке обучим Logistic Regression, Linear SVM и RBF SVM, подберём гиперпараметры и сравним качество моделей с помощью кросс-валидации.

**Основные задачи:**

1. Определить метрики качества и стратегию кросс-валидации с учётом дисбаланса классов.
2. Обучить базовые модели и оценить их качество.
3. Подобрать гиперпараметры Logistic Regression, Linear SVM и RBF SVM.
4. Сравнить комбинированное кодирование категориальных признаков с One-Hot Encoding.
5. Выбрать лучшие конфигурации моделей для итоговой оценки на тестовой выборке.

## 1. Подготовка данных

Загрузим исходный датасет, воспроизведём разделение на обучающую и тестовую выборки и подключим пайплайн предобработки из `src/preprocessing.py`.

При обучении моделей будем использовать исходные признаки, чтобы все преобразования выполнялись внутри кросс-валидации.

In [1]:
import sys
from pathlib import Path
import joblib

import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    make_scorer,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Корневая папка проекта
project_root = Path.cwd().resolve()

if project_root.name == 'notebooks':
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preprocessing import (
    build_preprocessing_pipeline,
    transform_pdays,
    NUMERIC_COLS,
    ONEHOT_COLS,
    TARGET_COLS,
    CYCLICAL_COLS,
    BINARY_COLS
)

In [2]:
# Загрузка данных
df = pd.read_csv(project_root / 'data' / 'raw' / 'banking.csv')

# Признаки и целевая переменная
X = df.drop(columns=['y', 'duration'])
y = df['y'].copy()

# Разделение данных
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

X_train: (32950, 19)
X_test: (8238, 19)
y_train: (32950,)
y_test: (8238,)


## 2. Стратегия оценки моделей

Определим метрики качества, настроим кросс-валидацию и оценим базовую модель.

Все эксперименты проведём на обучающей выборке. Тестовые данные сохраним для итогового сравнения.

### 2.1. Выбор метрик

Основной метрикой для подбора моделей выберем **Average Precision (AP)**, поскольку положительный класс представлен значительно реже отрицательного.

Дополнительно будем оценивать ROC-AUC, Precision, Recall и F1-score.

AP и ROC-AUC рассчитываются по непрерывным оценкам модели и не требуют выбора порога классификации. Precision, Recall и F1-score на этом этапе рассчитываются по стандартным предсказаниям классификаторов.

In [3]:
scoring = {
    'AP': 'average_precision',
    'ROC-AUC': 'roc_auc',
    'Precision': make_scorer(precision_score, zero_division=0),
    'Recall': make_scorer(recall_score, zero_division=0),
    'F1': make_scorer(f1_score, zero_division=0)
}

In [4]:
def summarize_cv(results, scoring):
    return pd.DataFrame({
        'Metric': list(scoring),
        'Mean': [
            results[f'test_{metric}'].mean()
            for metric in scoring
        ],
        'Std': [
            results[f'test_{metric}'].std()
            for metric in scoring
        ]
    })

### 2.2. Настройка кросс-валидации

Для оценки моделей используем `StratifiedKFold` с пятью фолдами.

Стратификация позволяет сохранить соотношение классов в каждом фолде, а фиксированный `random_state` обеспечивает воспроизводимость результатов.

Для всех моделей будем использовать одинаковую схему кросс-валидации.

In [5]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

### 2.3. Baseline-модель

Для базового сравнения используем `DummyClassifier(strategy='prior')`.

Модель не учитывает признаки клиентов: её предсказания основаны на распределении целевой переменной в обучающих данных.

Результаты DummyClassifier будем использовать как ориентир для оценки Logistic Regression и SVM.

In [6]:
dummy_model = DummyClassifier(strategy='prior')

dummy_cv = cross_validate(
    dummy_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

dummy_results = summarize_cv(dummy_cv, scoring)

dummy_results.round(4)

,Metric,Mean,Std
0,AP,0.1127,0.0001
1,ROC-AUC,0.5000,0.0000
2,Precision,0.0000,0.0000
3,Recall,0.0000,0.0000
4,F1,0.0000,0.0000


In [7]:
cv_results = {}

cv_results['DummyClassifier'] = {
    metric: dummy_cv[f'test_{metric}'].mean()
    for metric in scoring
}

### Выводы

`DummyClassifier` показал AP = 0.1127 и ROC-AUC = 0.5. Модель не обнаружила ни одного положительного наблюдения (Recall = 0).

Полученные результаты будем использовать как базовый ориентир для сравнения Logistic Regression и SVM.

## 3. Logistic Regression

Логистическая регрессия будет первой моделью бинарной классификации.

Сначала оценим базовую модель с помощью кросс-валидации, затем подберём параметры регуляризации и веса классов.

Предобработку включим в общий `Pipeline`, чтобы избежать утечки данных между обучающими и валидационными фолдами.

### 3.1. Обучение базовой модели

Создадим пайплайн с логистической регрессией и оценим его качество на пяти фолдах стратифицированной кросс-валидации.

Используем стандартные параметры классификатора, увеличив `max_iter` для обеспечения сходимости.

In [8]:
lr_pipeline = Pipeline([
    ('preprocessing', build_preprocessing_pipeline()),
    ('model', LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

In [9]:
lr_cv = cross_validate(
    lr_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

lr_results = summarize_cv(lr_cv, scoring)

lr_results.round(4)

,Metric,Mean,Std
0,AP,0.4506,0.0195
1,ROC-AUC,0.7824,0.0092
2,Precision,0.6614,0.0303
3,Recall,0.2255,0.0114
4,F1,0.3362,0.0159


In [10]:
baseline_comparison = pd.DataFrame({
    'DummyClassifier': dummy_results.set_index('Metric')['Mean'],
    'Logistic Regression': lr_results.set_index('Metric')['Mean']
})

baseline_comparison.round(4)

,DummyClassifier,Logistic Regression
Metric,,
AP,0.1127,0.4506
ROC-AUC,0.5000,0.7824
Precision,0.0000,0.6614
Recall,0.0000,0.2255
F1,0.0000,0.3362


### 3.2. Подбор гиперпараметров

Для настройки логистической регрессии используем `GridSearchCV`.

Подберём коэффициент регуляризации `C` и параметр `class_weight`, отвечающий за веса классов.

Основной метрикой выбора гиперпараметров будет Average Precision. Все комбинации оценим на одинаковых фолдах кросс-валидации.

In [11]:
lr_param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__class_weight': [None, 'balanced']
}

In [12]:
lr_grid = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    scoring=scoring,
    refit='AP',
    cv=cv,
    n_jobs=-1
)

In [13]:
lr_grid.fit(X_train, y_train)

print('Лучшие параметры:', lr_grid.best_params_)
print('Лучший AP:', round(lr_grid.best_score_, 4))

Лучшие параметры: {'model__C': 0.1, 'model__class_weight': None}
Лучший AP: 0.4514


### 3.3. Результаты

Сравним качество логистической регрессии при разных значениях `C` и `class_weight`.

Определим лучшую конфигурацию по Average Precision и сохраним её для дальнейшего сравнения с SVM.

In [14]:
lr_grid_results = pd.DataFrame(lr_grid.cv_results_)

lr_grid_results = lr_grid_results[[
    'param_model__C',
    'param_model__class_weight',
    'mean_test_AP',
    'std_test_AP',
    'mean_test_ROC-AUC',
    'mean_test_Precision',
    'mean_test_Recall',
    'mean_test_F1'
]]

lr_grid_results['param_model__class_weight'] = (
    lr_grid_results['param_model__class_weight']
    .fillna('None')
)

lr_grid_results = lr_grid_results.sort_values(
    by='mean_test_AP',
    ascending=False
)

lr_grid_results.round(4)

,param_model__C,param_model__class_weight,mean_test_AP,std_test_AP,mean_test_ROC-AUC,mean_test_Precision,mean_test_Recall,mean_test_F1
2,0.10,None,0.4514,0.0187,0.7822,0.6667,0.2209,0.3318
4,1.00,None,0.4506,0.0195,0.7824,0.6614,0.2255,0.3362
6,10.00,None,0.4504,0.0195,0.7824,0.6632,0.2271,0.3382
8,100.00,None,0.4503,0.0195,0.7824,0.6643,0.2271,0.3384
3,0.10,balanced,0.4479,0.0176,0.7847,0.2991,0.6565,0.4108
7,10.00,balanced,0.4475,0.0183,0.7846,0.3002,0.6565,0.4119
5,1.00,balanced,0.4475,0.0181,0.7846,0.3000,0.6565,0.4117
9,100.00,balanced,0.4475,0.0183,0.7846,0.3002,0.6565,0.4119
0,0.01,None,0.4434,0.0151,0.7795,0.6906,0.2010,0.3112
1,0.01,balanced,0.4402,0.0143,0.7836,0.2948,0.6616,0.4077


In [15]:
best_lr = lr_grid.best_estimator_

best_idx = lr_grid.best_index_

cv_results['Logistic Regression'] = {
    metric: lr_grid.cv_results_[f'mean_test_{metric}'][best_idx]
    for metric in scoring
}

### Выводы

- Базовая Logistic Regression показала AP = 0.4506 и ROC-AUC = 0.7824, значительно превзойдя DummyClassifier.
- Лучший AP (0.4514) получен при `C=0.1` и `class_weight=None`. При дальнейшем увеличении `C` качество практически не меняется, а при `C=0.01` несколько снижается.
- Использование `class_weight='balanced'` увеличило Recall примерно с 0.22 до 0.66, но снизило Precision с 0.67 до 0.30. AP при этом изменился незначительно.
- Лучшая конфигурация по AP сохранена для дальнейшего сравнения с SVM.

## 4. Linear SVM

Linear SVM — линейный классификатор, который ищет разделяющую гиперплоскость с максимальным отступом между классами.

В отличие от Logistic Regression, модель обучается с использованием hinge-подобной функции потерь и не оценивает вероятности напрямую.

Сначала оценим базовую модель, затем подберём коэффициент регуляризации `C` и веса классов.

### 4.1. Обучение базовой модели

Создадим пайплайн с `LinearSVC` и оценим его качество с помощью пятифолдовой стратифицированной кросс-валидации.

Используем стандартную L2-регуляризацию и `squared_hinge` в качестве функции потерь.

In [16]:
linear_svm_pipeline = Pipeline([
    ('preprocessing', build_preprocessing_pipeline()),
    ('model', LinearSVC(
        C=1.0,
        loss='squared_hinge',
        dual=False,
        max_iter=5000
    ))
])

In [17]:
linear_svm_cv = cross_validate(
    linear_svm_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

In [18]:
linear_svm_results = summarize_cv(linear_svm_cv, scoring)

linear_svm_results.round(4)

,Metric,Mean,Std
0,AP,0.4463,0.0189
1,ROC-AUC,0.7782,0.0083
2,Precision,0.7064,0.0235
3,Recall,0.1915,0.0082
4,F1,0.3013,0.0112


### 4.2. Подбор гиперпараметров

С помощью `GridSearchCV` подберём коэффициент регуляризации `C` и параметр `class_weight`.

Все комбинации оценим на одинаковых фолдах кросс-валидации. Основной метрикой выбора останется Average Precision.

In [19]:
linear_svm_param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__class_weight': [None, 'balanced']
}

In [20]:
linear_svm_grid = GridSearchCV(
    estimator=linear_svm_pipeline,
    param_grid=linear_svm_param_grid,
    scoring=scoring,
    refit='AP',
    cv=cv,
    n_jobs=2
)

In [21]:
linear_svm_grid.fit(X_train, y_train)

print('Лучшие параметры:', linear_svm_grid.best_params_)
print('Лучший AP:', round(linear_svm_grid.best_score_, 4))

Лучшие параметры: {'model__C': 0.01, 'model__class_weight': None}
Лучший AP: 0.4464


### 4.3. Результаты

Сравним качество Linear SVM при разных значениях `C` и `class_weight`.

Определим лучшую конфигурацию по Average Precision и сопоставим её с результатами Logistic Regression.

In [22]:
linear_svm_grid_results = pd.DataFrame(
    linear_svm_grid.cv_results_
)

linear_svm_grid_results = linear_svm_grid_results[[
    'param_model__C',
    'param_model__class_weight',
    'mean_test_AP',
    'std_test_AP',
    'mean_test_ROC-AUC',
    'mean_test_Precision',
    'mean_test_Recall',
    'mean_test_F1'
]]

linear_svm_grid_results['param_model__class_weight'] = (
    linear_svm_grid_results['param_model__class_weight']
    .fillna('None')
)

linear_svm_grid_results = linear_svm_grid_results.sort_values(
    by='mean_test_AP',
    ascending=False
)

linear_svm_grid_results.round(4)

,param_model__C,param_model__class_weight,mean_test_AP,std_test_AP,mean_test_ROC-AUC,mean_test_Precision,mean_test_Recall,mean_test_F1
0,0.01,None,0.4464,0.0181,0.7781,0.7056,0.1883,0.2972
8,100.00,None,0.4463,0.0189,0.7782,0.7064,0.1915,0.3013
4,1.00,None,0.4463,0.0189,0.7782,0.7064,0.1915,0.3013
6,10.00,None,0.4462,0.0190,0.7782,0.7064,0.1915,0.3013
2,0.10,None,0.4462,0.0188,0.7782,0.7085,0.1915,0.3015
1,0.01,balanced,0.4440,0.0187,0.7839,0.2914,0.6608,0.4044
3,0.10,balanced,0.4433,0.0190,0.7840,0.2923,0.6606,0.4052
7,10.00,balanced,0.4432,0.0188,0.7840,0.2924,0.6603,0.4052
5,1.00,balanced,0.4432,0.0188,0.7840,0.2924,0.6603,0.4052
9,100.00,balanced,0.4432,0.0188,0.7840,0.2924,0.6603,0.4052


In [23]:
best_linear_svm = linear_svm_grid.best_estimator_

best_idx = linear_svm_grid.best_index_

cv_results['Linear SVM'] = {
    metric: linear_svm_grid.cv_results_[f'mean_test_{metric}'][best_idx]
    for metric in scoring
}

In [24]:
linear_models_comparison = pd.DataFrame({
    'Logistic Regression': {
        metric: lr_grid.cv_results_[f'mean_test_{metric}'][
            lr_grid.best_index_
        ]
        for metric in scoring
    },
    'Linear SVM': {
        metric: linear_svm_grid.cv_results_[f'mean_test_{metric}'][
            linear_svm_grid.best_index_
        ]
        for metric in scoring
    }
})

linear_models_comparison.round(4)

,Logistic Regression,Linear SVM
AP,0.4514,0.4464
ROC-AUC,0.7822,0.7781
Precision,0.6667,0.7056
Recall,0.2209,0.1883
F1,0.3318,0.2972


### Выводы

- Базовая Linear SVM показала AP = 0.4463 и ROC-AUC = 0.7782, немного уступив Logistic Regression.
- Лучший AP (0.4464) получен при `C=0.01` и `class_weight=None`. Подбор гиперпараметров практически не улучшил качество модели.
- Изменение `C` в исследованном диапазоне почти не повлияло на AP. Использование `class_weight='balanced'` увеличило Recall примерно с 0.19 до 0.66, но снизило Precision с 0.71 до 0.29.
- Logistic Regression показала более высокий AP (0.4514 против 0.4464), однако различие невелико относительно разброса результатов между фолдами.
- Лучшая конфигурация Linear SVM сохранена для дальнейшего сравнения с RBF SVM.

## 5. RBF SVM

В отличие от Logistic Regression и Linear SVM, RBF SVM позволяет строить нелинейную границу между классами за счёт использования радиальной базисной функции (RBF).

Проверим, улучшает ли нелинейность качество классификации. Сначала оценим базовую модель, затем подберём параметры `C`, `gamma` и `class_weight`.

### 5.1. Обучение базовой модели

Создадим пайплайн с `SVC(kernel='rbf')` и оценим его качество с помощью пятифолдовой стратифицированной кросс-валидации.

Используем стандартные значения `C=1.0` и `gamma='scale'`. Дополнительно оценим время обучения, поскольку RBF SVM вычислительно сложнее линейных моделей.

In [25]:
rbf_svm_pipeline = Pipeline([
    ('preprocessing', build_preprocessing_pipeline()),
    ('model', SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        cache_size=512
    ))
])

In [26]:
rbf_svm_cv = cross_validate(
    rbf_svm_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

In [27]:
rbf_svm_results = summarize_cv(rbf_svm_cv, scoring)

rbf_svm_results.round(4)

,Metric,Mean,Std
0,AP,0.4025,0.0175
1,ROC-AUC,0.6881,0.0037
2,Precision,0.6617,0.0273
3,Recall,0.2371,0.0154
4,F1,0.3488,0.0179


In [28]:
print(
    'Среднее время обучения одного фолда:',
    round(rbf_svm_cv['fit_time'].mean(), 2),
    'сек.'
)

Среднее время обучения одного фолда: 11.39 сек.


### 5.2. Подбор гиперпараметров

Для настройки RBF SVM используем `GridSearchCV`.

Подберём коэффициент регуляризации `C`, параметр ядра `gamma` и веса классов `class_weight`.

Основной метрикой выбора останется Average Precision. Для всех комбинаций используем ту же схему кросс-валидации, что и для предыдущих моделей.

In [29]:
rbf_svm_param_grid = {
    'model__C': [0.1, 1, 10],
    'model__gamma': [0.001, 0.01, 'scale', 0.1],
    'model__class_weight': [None, 'balanced']
}

In [30]:
rbf_svm_grid = GridSearchCV(
    estimator=rbf_svm_pipeline,
    param_grid=rbf_svm_param_grid,
    scoring=scoring,
    refit='AP',
    cv=cv,
    n_jobs=1,
    verbose=1
)

In [31]:
rbf_svm_grid.fit(X_train, y_train)

print('Лучшие параметры:', rbf_svm_grid.best_params_)
print('Лучший AP:', round(rbf_svm_grid.best_score_, 4))

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Лучшие параметры: {'model__C': 10, 'model__class_weight': 'balanced', 'model__gamma': 0.001}
Лучший AP: 0.4071


### 5.3. Результаты

Сравним результаты всех проверенных комбинаций гиперпараметров и определим лучшую конфигурацию RBF SVM по Average Precision.

Затем сопоставим её качество с результатами Logistic Regression и Linear SVM.

In [32]:
rbf_svm_grid_results = pd.DataFrame(
    rbf_svm_grid.cv_results_
)

rbf_svm_grid_results = rbf_svm_grid_results[[
    'param_model__C',
    'param_model__gamma',
    'param_model__class_weight',
    'mean_test_AP',
    'std_test_AP',
    'mean_test_ROC-AUC',
    'mean_test_Precision',
    'mean_test_Recall',
    'mean_test_F1'
]]

rbf_svm_grid_results['param_model__class_weight'] = (
    rbf_svm_grid_results['param_model__class_weight']
    .fillna('None')
)

rbf_svm_grid_results = rbf_svm_grid_results.sort_values(
    by='mean_test_AP',
    ascending=False
)

rbf_svm_grid_results.round(4)

,param_model__C,param_model__gamma,param_model__class_weight,mean_test_AP,std_test_AP,mean_test_ROC-AUC,mean_test_Precision,mean_test_Recall,mean_test_F1
20,10.0,0.001,balanced,0.4071,0.0132,0.7761,0.2457,0.7107,0.3651
17,10.0,0.01,None,0.4058,0.0204,0.6889,0.6975,0.1932,0.3023
10,1.0,scale,None,0.4025,0.0175,0.6881,0.6617,0.2371,0.3488
3,0.1,0.1,None,0.4016,0.0192,0.7092,0.7210,0.1404,0.2348
2,0.1,scale,None,0.4015,0.0202,0.7048,0.7092,0.1789,0.2856
9,1.0,0.01,None,0.4011,0.0212,0.6892,0.6865,0.1994,0.3089
12,1.0,0.001,balanced,0.4001,0.0153,0.7673,0.2448,0.7115,0.3643
5,0.1,0.01,balanced,0.3995,0.0135,0.7780,0.2459,0.7099,0.3653
4,0.1,0.001,balanced,0.3992,0.0149,0.7727,0.2448,0.7115,0.3643
11,1.0,0.1,None,0.3984,0.0182,0.6996,0.6582,0.2438,0.3556


In [33]:
best_rbf_svm = rbf_svm_grid.best_estimator_

best_idx = rbf_svm_grid.best_index_

cv_results['RBF SVM'] = {
    metric: rbf_svm_grid.cv_results_[f'mean_test_{metric}'][best_idx]
    for metric in scoring
}

### Выводы

- Базовая RBF SVM показала AP = 0.4025 и ROC-AUC = 0.6881, уступив линейным моделям.
- После подбора гиперпараметров лучший AP увеличился до 0.4071 при `C=10`, `gamma=0.001` и `class_weight='balanced'`.
- Настройка существенно повысила ROC-AUC (с 0.6881 до 0.7761) и Recall (с 0.24 до 0.71), но снизила Precision (с 0.66 до 0.25). Улучшение AP оказалось незначительным.
- Logistic Regression сохранила преимущество по основной метрике: AP = 0.4514 против 0.4071 у RBF SVM.
- В рамках исследованных конфигураций RBF-ядро не улучшило качество ранжирования по сравнению с линейными классификаторами.

## 6. Эксперимент с кодированием признаков

На этапе предобработки мы выбрали комбинированный подход: Target Encoding для `job` и `education`, циклическое кодирование для `month` и One-Hot Encoding для остальных категориальных признаков.

Проверим, улучшает ли такой подход качество классификации по сравнению с использованием One-Hot Encoding для всех категориальных признаков.

Для эксперимента используем Logistic Regression с ранее подобранными гиперпараметрами. Оба варианта предобработки оценим на одинаковых фолдах кросс-валидации.

### 6.1. Подготовка альтернативного пайплайна

Создадим альтернативный пайплайн, в котором все категориальные признаки обрабатываются через `OneHotEncoder`.

Масштабирование числовых признаков и преобразование `pdays` оставим без изменений, чтобы сравнение было сосредоточено на способах кодирования.

In [34]:
def build_onehot_preprocessing_pipeline():
    categorical_cols = ONEHOT_COLS + TARGET_COLS + CYCLICAL_COLS

    columns = ColumnTransformer(
        transformers=[
            ('numeric', StandardScaler(), NUMERIC_COLS),
            ('onehot', OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ), categorical_cols),
            ('binary', 'passthrough', BINARY_COLS)
        ],
        remainder='drop'
    )

    return Pipeline([
        ('pdays', FunctionTransformer(transform_pdays, validate=False)),
        ('columns', columns)
    ])

### 6.2. Сравнение способов кодирования

Сравним комбинированное кодирование с One-Hot Encoding на Logistic Regression.

Для обоих вариантов используем ранее выбранные гиперпараметры (`C=0.1`, `class_weight=None`) и одинаковую схему кросс-валидации.

Дополнительно сравним количество признаков после преобразования и время обучения.

In [35]:
combined_lr = clone(best_lr)

onehot_lr = Pipeline([
    ('preprocessing', build_onehot_preprocessing_pipeline()),
    ('model', clone(best_lr.named_steps['model']))
])

In [36]:
combined_cv = cross_validate(
    combined_lr,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_estimator=True
)

onehot_cv = cross_validate(
    onehot_lr,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_estimator=True
)

### 6.3. Результаты эксперимента

Сравним значения метрик, количество признаков после преобразования и среднее время обучения одного фолда.

Основным критерием качества останется Average Precision.

In [37]:
encoding_comparison = pd.DataFrame({
    'Combined': {
        metric: combined_cv[f'test_{metric}'].mean()
        for metric in scoring
    },
    'One-Hot': {
        metric: onehot_cv[f'test_{metric}'].mean()
        for metric in scoring
    }
})

encoding_comparison.round(4)

,Combined,One-Hot
AP,0.4514,0.4561
ROC-AUC,0.7822,0.7917
Precision,0.6667,0.6759
Recall,0.2209,0.2233
F1,0.3318,0.3356


In [38]:
encoding_complexity = pd.DataFrame({
    'Encoding': ['Combined', 'One-Hot'],
    'Features': [
        combined_cv['estimator'][0].named_steps['model'].coef_.shape[1],
        onehot_cv['estimator'][0].named_steps['model'].coef_.shape[1]
    ],
    'Mean fit time (s)': [
        combined_cv['fit_time'].mean(),
        onehot_cv['fit_time'].mean()
    ]
})

encoding_complexity.round(2)

,Encoding,Features,Mean fit time (s)
0,Combined,37,0.13
1,One-Hot,63,0.19


In [39]:
ap_diff = onehot_cv['test_AP'] - combined_cv['test_AP']

print('Разница AP по фолдам:', ap_diff.round(4))
print('Средняя разница:', round(ap_diff.mean(), 4))

Разница AP по фолдам: [0.0075 0.0014 0.0093 0.0019 0.0034]
Средняя разница: 0.0047


В первоначальном сравнении использовались гиперпараметры, подобранные для комбинированного кодирования.

Поскольку One-Hot Encoding показал более высокий AP, дополнительно подберём для него `C` и `class_weight` с помощью `GridSearchCV`, сохранив прежнюю стратегию кросс-валидации.

In [40]:
onehot_grid = GridSearchCV(
    estimator=onehot_lr,
    param_grid={
        'model__C': [0.01, 0.1, 1, 10, 100],
        'model__class_weight': [None, 'balanced']
    },
    scoring=scoring,
    refit='AP',
    cv=cv,
    n_jobs=2
)

onehot_grid.fit(X_train, y_train)

print('Лучшие параметры:', onehot_grid.best_params_)
print('Лучший AP:', round(onehot_grid.best_score_, 4))

Лучшие параметры: {'model__C': 10, 'model__class_weight': None}
Лучший AP: 0.4567


In [41]:
onehot_grid_results = pd.DataFrame(onehot_grid.cv_results_)

onehot_grid_results = onehot_grid_results[[
    'param_model__C',
    'param_model__class_weight',
    'mean_test_AP',
    'mean_test_ROC-AUC',
    'mean_test_Precision',
    'mean_test_Recall',
    'mean_test_F1'
]]

onehot_grid_results['param_model__class_weight'] = (
    onehot_grid_results['param_model__class_weight'].fillna('None')
)

onehot_grid_results.sort_values(
    'mean_test_AP', ascending=False
).round(4)

,param_model__C,param_model__class_weight,mean_test_AP,mean_test_ROC-AUC,mean_test_Precision,mean_test_Recall,mean_test_F1
6,10.00,None,0.4567,0.7912,0.6661,0.2320,0.3438
8,100.00,None,0.4567,0.7911,0.6659,0.2322,0.3441
4,1.00,None,0.4565,0.7913,0.6682,0.2303,0.3423
2,0.10,None,0.4561,0.7917,0.6759,0.2233,0.3356
5,1.00,balanced,0.4523,0.7922,0.3577,0.6282,0.4558
9,100.00,balanced,0.4522,0.7922,0.3583,0.6282,0.4563
7,10.00,balanced,0.4521,0.7922,0.3580,0.6282,0.4560
3,0.10,balanced,0.4520,0.7923,0.3557,0.6312,0.4549
0,0.01,None,0.4491,0.7893,0.6976,0.2031,0.3145
1,0.01,balanced,0.4448,0.7913,0.3386,0.6366,0.4420


In [42]:
best_onehot_lr = onehot_grid.best_estimator_

best_idx = onehot_grid.best_index_

cv_results['Logistic Regression (One-Hot)'] = {
    metric: onehot_grid.cv_results_[f'mean_test_{metric}'][best_idx]
    for metric in scoring
}


### Выводы

- При одинаковых гиперпараметрах One-Hot Encoding показал более высокий AP (0.4561 против 0.4514) и ROC-AUC (0.7917 против 0.7822). Преимущество по AP наблюдалось на всех пяти фолдах.

- Комбинированное кодирование сократило количество признаков с 63 до 37. Среднее время обучения одного фолда составило 0.13 с против 0.19 с для One-Hot Encoding.

- После дополнительного подбора гиперпараметров Logistic Regression с One-Hot Encoding достигла AP = 0.4567. Улучшение относительно первоначальной конфигурации оказалось минимальным.
- One-Hot Encoding выглядит более перспективным по качеству ранжирования, однако разница с комбинированным подходом невелика и требует подтверждения на независимой тестовой выборке.


### 6.4. Сравнение алгоритмов при одинаковом кодировании

Ранее Logistic Regression показала лучший результат с One-Hot Encoding, тогда как SVM обучались с комбинированным кодированием.

Чтобы сравнить алгоритмы в одинаковых условиях, обучим Linear SVM и RBF SVM с One-Hot Encoding. Сохраним прежние фолды кросс-валидации и сетки гиперпараметров.

#### Linear SVM с One-Hot Encoding

Используем тот же Linear SVM, но заменим предобработку на One-Hot Encoding. Гиперпараметры подберём заново по Average Precision.

In [43]:
# Linear SVM с One-Hot Encoding
linear_onehot_pipeline = clone(linear_svm_pipeline)

linear_onehot_pipeline.set_params(
    preprocessing=build_onehot_preprocessing_pipeline()
)

# Подбор гиперпараметров
linear_onehot_grid = GridSearchCV(
    estimator=linear_onehot_pipeline,
    param_grid=linear_svm_param_grid,
    scoring=scoring,
    refit='AP',
    cv=cv,
    n_jobs=2
)

linear_onehot_grid.fit(X_train, y_train)

print('Лучшие параметры:', linear_onehot_grid.best_params_)
print('Лучший CV AP:', round(linear_onehot_grid.best_score_, 4))

best_idx = linear_onehot_grid.best_index_

print(
    'CV ROC-AUC:',
    round(
        linear_onehot_grid.cv_results_['mean_test_ROC-AUC'][best_idx],
        4
    )
)


Лучшие параметры: {'model__C': 0.1, 'model__class_weight': None}
Лучший CV AP: 0.4518
CV ROC-AUC: 0.7876


#### RBF SVM с One-Hot Encoding

Проверим, изменится ли качество RBF SVM при использовании One-Hot Encoding. Для сравнения оставим прежнюю сетку гиперпараметров и схему кросс-валидации.

In [44]:
# RBF SVM с One-Hot Encoding
rbf_onehot_pipeline = clone(rbf_svm_pipeline)

rbf_onehot_pipeline.set_params(
    preprocessing=build_onehot_preprocessing_pipeline()
)

# Подбор гиперпараметров
rbf_onehot_grid = GridSearchCV(
    estimator=rbf_onehot_pipeline,
    param_grid=rbf_svm_param_grid,
    scoring=scoring,
    refit='AP',
    cv=cv,
    n_jobs=1,
    verbose=1
)

rbf_onehot_grid.fit(X_train, y_train)

print('Лучшие параметры:', rbf_onehot_grid.best_params_)
print('Лучший CV AP:', round(rbf_onehot_grid.best_score_, 4))

best_idx = rbf_onehot_grid.best_index_

print(
    'CV ROC-AUC:',
    round(
        rbf_onehot_grid.cv_results_['mean_test_ROC-AUC'][best_idx],
        4
    )
)


Fitting 5 folds for each of 24 candidates, totalling 120 fits
Лучшие параметры: {'model__C': 10, 'model__class_weight': 'balanced', 'model__gamma': 0.001}
Лучший CV AP: 0.4167
CV ROC-AUC: 0.7769


#### Сравнение результатов

Сравним Logistic Regression, Linear SVM и RBF SVM при двух способах кодирования признаков. Основная метрика — Average Precision.


In [45]:
# Добавляем новые конфигурации в общие результаты CV
new_grids = {
    'Linear SVM (One-Hot)': linear_onehot_grid,
    'RBF SVM (One-Hot)': rbf_onehot_grid
}

for name, grid in new_grids.items():
    best_idx = grid.best_index_

    cv_results[name] = {
        metric: grid.cv_results_[f'mean_test_{metric}'][best_idx]
        for metric in scoring
    }

# Сравнение всех конфигураций
encoding_models_comparison = (
    pd.DataFrame.from_dict(cv_results, orient='index')
    .sort_values('AP', ascending=False)
)

encoding_models_comparison.round(4)


,AP,ROC-AUC,Precision,Recall,F1
Logistic Regression (One-Hot),0.4567,0.7912,0.6661,0.2320,0.3438
Linear SVM (One-Hot),0.4518,0.7876,0.6814,0.2152,0.3270
Logistic Regression,0.4514,0.7822,0.6667,0.2209,0.3318
Linear SVM,0.4464,0.7781,0.7056,0.1883,0.2972
RBF SVM (One-Hot),0.4167,0.7769,0.3640,0.6156,0.4574
RBF SVM,0.4071,0.7761,0.2457,0.7107,0.3651
DummyClassifier,0.1127,0.5000,0.0000,0.0000,0.0000


#### Выводы

One-Hot Encoding улучшил Average Precision у всех трёх алгоритмов. Для Linear SVM AP выросла с 0.4464 до 0.4518, для RBF SVM — с 0.4071 до 0.4167.

Лучший результат сохранила Logistic Regression с One-Hot Encoding (AP = 0.4567). Linear SVM с той же предобработкой показала близкое качество (AP = 0.4518), тогда как RBF SVM заметно уступила обеим моделям.

Таким образом, различия в первоначальном сравнении были частично связаны со способом кодирования признаков. При одинаковой предобработке преимущество Logistic Regression над Linear SVM оказалось небольшим.



## 7. Итоги моделирования

На этапе моделирования были обучены Logistic Regression, Linear SVM и RBF SVM, выполнен подбор гиперпараметров и проведён эксперимент с различными способами кодирования категориальных признаков.

Соберём результаты кросс-валидации, определим наиболее перспективные конфигурации и сохраним обученные пайплайны для итоговой оценки на тестовой выборке.



### 7.1. Сводная таблица результатов

Сравним лучшие конфигурации моделей по результатам кросс-валидации на обучающей выборке.

Основным критерием остаётся Average Precision. Дополнительные метрики позволяют оценить различия в ранжировании и классификации при стандартных порогах.


In [46]:
model_comparison = (
    pd.DataFrame.from_dict(cv_results, orient='index')
    .sort_values('AP', ascending=False)
)

model_comparison.round(4)

,AP,ROC-AUC,Precision,Recall,F1
Logistic Regression (One-Hot),0.4567,0.7912,0.6661,0.2320,0.3438
Linear SVM (One-Hot),0.4518,0.7876,0.6814,0.2152,0.3270
Logistic Regression,0.4514,0.7822,0.6667,0.2209,0.3318
Linear SVM,0.4464,0.7781,0.7056,0.1883,0.2972
RBF SVM (One-Hot),0.4167,0.7769,0.3640,0.6156,0.4574
RBF SVM,0.4071,0.7761,0.2457,0.7107,0.3651
DummyClassifier,0.1127,0.5000,0.0000,0.0000,0.0000


In [47]:
# Подготовка результатов для следующего ноутбука
cv_reference = (
    model_comparison
    .drop(index='DummyClassifier')
    .rename(index={
        'Logistic Regression': 'Logistic Regression (Combined)',
        'Linear SVM': 'Linear SVM (Combined)',
        'RBF SVM': 'RBF SVM (Combined)'
    })
    [['AP', 'ROC-AUC']]
)

# Сохранение результатов
results_dir = project_root / 'results'
results_dir.mkdir(exist_ok=True)

cv_reference.to_csv(results_dir / 'cv_results.csv')


### 7.2. Выбор моделей для итогового сравнения

Наибольший Average Precision показала Logistic Regression с One-Hot Encoding (0.4567). Linear SVM с таким же кодированием получила близкий результат (0.4518), тогда как RBF SVM заметно уступила линейным моделям (0.4167).

One-Hot Encoding улучшил качество всех трёх алгоритмов. При одинаковой предобработке разница между Logistic Regression и Linear SVM составила всего 0.0049 AP.

Для дальнейшего сравнения сохраним все шесть конфигураций моделей. Результаты кросс-валидации использовались для подбора параметров, поэтому для дополнительной оценки рассмотрим их качество на Test.



### 7.3. Сохранение обученных моделей

Сохраним лучшие пайплайны с помощью `joblib`.

Каждый пайплайн включает предобработку и обученный классификатор. Это позволит использовать модели в следующем ноутбуке без повторного обучения.

Тестовая выборка не использовалась при подборе гиперпараметров или обучении сохранённых моделей.


In [48]:
models_dir = project_root / 'models'
models_dir.mkdir(exist_ok=True)

models = {
    'logistic_combined': best_lr,
    'logistic_onehot': best_onehot_lr,
    'linear_svm': best_linear_svm,
    'linear_svm_onehot': linear_onehot_grid.best_estimator_,
    'rbf_svm': best_rbf_svm,
    'rbf_svm_onehot': rbf_onehot_grid.best_estimator_
}

for name, model in models.items():
    joblib.dump(model, models_dir / f'{name}.joblib')

print(f'Сохранено моделей: {len(models)}')


Сохранено моделей: 6


### Итог

При одинаковом One-Hot Encoding лучший результат сохранила Logistic Regression (AP = 0.4567). Linear SVM показала близкое качество (AP = 0.4518), а RBF SVM заметно уступила обеим моделям.

Способ кодирования повлиял на качество всех алгоритмов, поэтому первоначальное сравнение было не совсем равнозначным.

Все шесть конфигураций обучены на Train и сохранены для дальнейшей оценки.